In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import duckdb

from ga4case.path import COMPACT_DATA, EVENTS_FLATTENED

In [3]:
con = duckdb.connect()

In [4]:
# Check on the raw data file in data/raw_data/
# con.sql("""
#         select count(distinct row_group_id) as row_groups,
#                 count(*) as index_entries
#         from parquet_metadata('../data/raw_data/ga4_events.parquet')
#         """)

In [5]:
DATA = COMPACT_DATA.as_posix()

con.sql(f"""
        select count(distinct row_group_id) as row_groups,
                count(*) as index_entries
        from parquet_metadata('{DATA}') 
        """)

row_groupsBIGINT,index_entriesBIGINT
35,3290


In [6]:
con.sql(f"""
        select * 
        from read_parquet('{DATA}')
        limit 2
        """).df()

,event_date,event_timestamp,event_name,event_params,event_previous_timestamp,event_value_in_usd,event_bundle_sequence_id,event_server_timestamp_offset,user_id,user_pseudo_id,...,user_ltv,device,geo,app_info,traffic_source,stream_id,platform,event_dimensions,ecommerce,items
0,20210131,1612124024141525,select_item,"[{'key': 'engaged_session_event', 'value': {'s...",<NA>,NaN,1596965562,<NA>,None,4038731.3310220776,...,"{'revenue': 0.0, 'currency': 'USD'}","{'category': 'mobile', 'mobile_brand_name': 'A...","{'continent': 'Americas', 'sub_continent': 'No...",<NA>,"{'medium': 'organic', 'name': '(organic)', 'so...",2100450278,WEB,<NA>,"{'total_item_quantity': None, 'purchase_revenu...","[{'item_id': 'GGOEGXXX1038', 'item_name': 'Goo..."
1,20210131,1612103390989332,select_item,"[{'key': 'page_referrer', 'value': {'string_va...",<NA>,NaN,9961797504,<NA>,None,25631840.3215691016,...,"{'revenue': 0.0, 'currency': 'USD'}","{'category': 'desktop', 'mobile_brand_name': '...","{'continent': 'Americas', 'sub_continent': 'No...",<NA>,"{'medium': '<Other>', 'name': '<Other>', 'sour...",2100450278,WEB,<NA>,"{'total_item_quantity': None, 'purchase_revenu...","[{'item_id': 'GGOEGBRJ128599', 'item_name': 'S..."


In [7]:
# Check the number and columns of the events data
events = con.sql(f"""
        select *
        from read_parquet('{DATA}')
        """)
events.shape

(4295584, 23)

In [8]:
# check the columns/ nested columns
schema = con.sql(f"""
        describe select * from read_parquet('{DATA}')""").fetchall()
for name, coltype, *_ in schema:
    print(name, "-->", coltype,"\n")

event_date --> VARCHAR 

event_timestamp --> BIGINT 

event_name --> VARCHAR 

event_params --> STRUCT("key" VARCHAR, "value" STRUCT(string_value VARCHAR, int_value BIGINT, float_value DOUBLE, double_value DOUBLE))[] 

event_previous_timestamp --> BIGINT 

event_value_in_usd --> DOUBLE 

event_bundle_sequence_id --> BIGINT 

event_server_timestamp_offset --> BIGINT 

user_id --> VARCHAR 

user_pseudo_id --> VARCHAR 

privacy_info --> STRUCT(analytics_storage BIGINT, ads_storage BIGINT, uses_transient_token VARCHAR) 

user_properties --> STRUCT("key" BIGINT, "value" STRUCT(string_value BIGINT, int_value BIGINT, float_value BIGINT, double_value BIGINT, set_timestamp_micros BIGINT))[] 

user_first_touch_timestamp --> BIGINT 

user_ltv --> STRUCT(revenue DOUBLE, currency VARCHAR) 

device --> STRUCT(category VARCHAR, mobile_brand_name VARCHAR, mobile_model_name VARCHAR, mobile_marketing_name VARCHAR, mobile_os_hardware_model BIGINT, operating_system VARCHAR, operating_system_version VARCHA

## Step 1 Flat Events Table

In [9]:
flattened_sql = f"""
    with src as(
        select *, map_from_entries(event_params) as params
        from read_parquet('{DATA}')
    )
    select
        user_pseudo_id,
        make_timestamp(event_timestamp) as event_timestamp,
        strptime(event_date, '%Y%m%d')::date as event_date,
        event_name,
        params['entrances'].int_value as entrances,
        params['ga_session_id'].int_value as ga_session_id,
        params['ga_session_number'].int_value as ga_session_number,
        params['page_location'].string_value as page_location,
        params['page_referrer'].string_value as page_referrer,
        params['page_title'].string_value as page_title,
        params['engagement_time_msec'].int_value as engagement_time_msec,
        coalesce(params['session_engaged'].string_value, cast(params['session_engaged'].int_value as varchar)) as session_engaged,
        device.category as device_category,
        device.operating_system as device_os,
        device.web_info.browser as device_browser,
        geo.country as geo_country,
        traffic_source.source as traffic_source,
        traffic_source.medium as traffic_medium,
        ecommerce.purchase_revenue_in_usd as purchase_revenue_in_usd,
        nullif(ecommerce.transaction_id, '(not set)') as transaction_id 
    from src
    """

# About transaction_id: '(not set)' is a placeholder for missing values in GA4,  we will replace it with NULL, otherwise when counting distinct transaction_id, it will be counted as a valid transaction_id

In [10]:
con.sql(flattened_sql + "limit 10").df()


,user_pseudo_id,event_timestamp,event_date,event_name,entrances,ga_session_id,ga_session_number,page_location,page_referrer,page_title,engagement_time_msec,session_engaged,device_category,device_os,device_browser,geo_country,traffic_source,traffic_medium,purchase_revenue_in_usd,transaction_id
0,4038731.3310220776,2021-01-31 20:13:44.141525,2021-01-31,select_item,<NA>,5566381072,1,https://shop.googlemerchandisestore.com/asearc...,https://shop.googlemerchandisestore.com/store....,Store search results,1917,1,mobile,iOS,<Other>,United States,google,organic,NaN,None
1,25631840.3215691016,2021-01-31 14:29:50.989332,2021-01-31,select_item,<NA>,3289776494,4,https://shop.googlemerchandisestore.com/asearc...,https://shop.googlemerchandisestore.com/?,Store search results,3930,1,desktop,Web,Chrome,United States,<Other>,<Other>,NaN,None
2,34307529.8359866913,2021-01-13 00:56:22.561657,2021-01-13,first_visit,<NA>,6348737127,1,https://shop.googlemerchandisestore.com/,NaN,Home,<NA>,1,desktop,Macintosh,Safari,Vietnam,<Other>,referral,NaN,None
3,82208273.1273401149,2020-12-27 05:01:45.865686,2020-12-27,view_item,<NA>,4995201016,2,https://shop.googlemerchandisestore.com/Google...,https://shop.googlemerchandisestore.com/basket...,Apparel | Google Merchandise Store,4910,1,mobile,iOS,Safari,United Kingdom,(data deleted),(data deleted),NaN,None
4,56594632.3842138936,2020-12-27 18:56:50.397162,2020-12-27,view_promotion,<NA>,9295651384,2,https://shop.googlemerchandisestore.com/,https://shop.googlemerchandisestore.com/forgot...,Home,229,1,mobile,Web,Android Webview,Indonesia,google,organic,NaN,None
5,19281465.1734784980,2021-01-21 10:14:42.230025,2021-01-21,user_engagement,<NA>,3217216963,4,https://shop.googlemerchandisestore.com/asearc...,NaN,Store search results,3530,1,mobile,Web,Safari,Russia,google,organic,NaN,None
6,30284467.0158867020,2021-01-21 23:42:38.691819,2021-01-21,page_view,<NA>,4581377141,1,https://googlemerchandisestore.com/,NaN,Google Online Store,3,1,desktop,Macintosh,Chrome,United States,google,organic,NaN,None
7,31566750.6615222194,2021-01-21 20:52:13.568720,2021-01-21,user_engagement,<NA>,7826584646,1,https://shop.googlemerchandisestore.com/google...,NaN,Page Unavailable,1434,1,desktop,Windows,Chrome,Canada,google,cpc,NaN,None
8,22659306.8402858234,2021-01-27 19:30:32.024996,2021-01-27,user_engagement,<NA>,8151685918,3,https://shop.googlemerchandisestore.com/,NaN,Home,<NA>,0,desktop,Web,Chrome,Canada,<Other>,<Other>,NaN,None
9,35847520.1676036636,2021-01-27 18:28:20.027942,2021-01-27,scroll,<NA>,6540657730,1,https://www.googlemerchandisestore.com/,NaN,Google Online Store,433,1,mobile,Android,Chrome,United States,<Other>,<Other>,NaN,None


In [11]:
con.sql(f"""
        select
                (select count(*) from read_parquet('{DATA}')) as row_in_source,
                count(*) as row_in_flattened,
                count(distinct user_pseudo_id) as users,
                count(distinct(user_pseudo_id, ga_session_id)) as sessions,
                count(*) - count(ga_session_id) as rows_without_session_id,
                count(*) - count(distinct(user_pseudo_id, event_timestamp, event_name)) as duplicate_events
        from ({flattened_sql})
        """).df()

,row_in_source,row_in_flattened,users,sessions,rows_without_session_id,duplicate_events
0,4295584,4295584,270154,360129,0,0


In [12]:
con.sql(f"summarize select * from ({flattened_sql})").df()[
    ["column_name", "column_type", "null_percentage", "approx_unique"]
]

,column_name,column_type,null_percentage,approx_unique
0,user_pseudo_id,VARCHAR,0.00,258699
1,event_timestamp,TIMESTAMP,0.00,2872567
2,event_date,DATE,0.00,92
3,event_name,VARCHAR,0.00,17
4,entrances,BIGINT,92.27,1
5,ga_session_id,BIGINT,0.00,352485
6,ga_session_number,BIGINT,0.00,99
7,page_location,VARCHAR,0.00,2445
8,page_referrer,VARCHAR,73.75,361
9,page_title,VARCHAR,0.49,495


In [13]:
con.sql(f"""
    select
        min(event_timestamp) as first_event,
        max(event_timestamp) as last_event,
        count(distinct event_date) as days,
        count(distinct event_name) as event_types,
        sum((cast(event_timestamp as date)<> event_date)::int) as date_mismatches
    from ({flattened_sql})
    """).df()

,first_event,last_event,days,event_types,date_mismatches
0,2020-11-01 00:00:04.579566,2021-01-31 23:59:55.412363,92,17,0.0


1. The flattened table has the same number of rows equals to the row number in the source data, every event has a session id, and no event is duplicated.
2. This data consists of three levels:
    - User level: 270,154 different visitors (browsers: browser is not a person)
    - Session level: 360,129 visits by visitors
    - Event level: 4,295,584 single actions inside visits
3. Every column is filled as expected and none is completely empty.
4. Table records covered from Nov. 1st to Jan.31st: 92 days.